# Opik Metrics Explorer

Notebook para carregar traces e threads do Opik, escolher colunas/metricas e calcular agregacoes.

Baseado na API oficial do SDK:
- `client.search_traces(project_name=..., filter_string=..., max_results=..., truncate=..., exclude=...)`
- `client.search_threads(project_name=..., filter_string=..., max_results=..., truncate=...)`
- `client.log_threads_feedback_scores(scores=[...], project_name=...)` para feedback por thread

Os filtros suportam campos como `thread_id`, `tags`, `metadata.*`, `feedback_scores.*`, `usage.*`, `duration` e outros.

In [1]:
from __future__ import annotations

import os
from typing import Any, Iterable

import pandas as pd

try:
    from opik import Opik
except Exception as exc:
    raise RuntimeError("The opik package is required in this environment.") from exc

pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 200)
pd.set_option('display.max_colwidth', 200)

In [2]:
PROJECT_NAME = os.getenv('OPIK_PROJECT_NAME', 'ChatDev')
OPIK_HOST = os.getenv('OPIK_URL_OVERRIDE', 'https://www.comet.com/opik/api')
OPIK_WORKSPACE = os.getenv('OPIK_WORKSPACE') or os.getenv('OPIK_WORKSPACE_NAME')
OPIK_API_KEY = os.getenv('OPIK_API_KEY')

client = Opik(
    project_name=PROJECT_NAME,
    host=OPIK_HOST,
    workspace=OPIK_WORKSPACE,
    api_key=OPIK_API_KEY,
)

PROJECT_NAME

'ChatDev-AASMA'

In [3]:
def flatten_dict(value: dict[str, Any], prefix: str = '', sep: str = '.') -> dict[str, Any]:
    items: dict[str, Any] = {}
    for key, item in value.items():
        new_key = f'{prefix}{sep}{key}' if prefix else key
        if isinstance(item, dict):
            items.update(flatten_dict(item, new_key, sep=sep))
        else:
            items[new_key] = item
    return items

def pick_fields(record: Any, fields: Iterable[str]) -> dict[str, Any]:
    data: dict[str, Any] = {}
    for field in fields:
        value = record
        for part in field.split('.'):
            if value is None:
                break
            if isinstance(value, dict):
                value = value.get(part)
            else:
                value = getattr(value, part, None)
        data[field] = value
    return data

def as_dict(record: Any) -> dict[str, Any]:
    if isinstance(record, dict):
        return record
    if hasattr(record, 'model_dump'):
        return record.model_dump()
    if hasattr(record, 'dict'):
        return record.dict()
    return {k: getattr(record, k) for k in dir(record) if not k.startswith('_')}

In [4]:
def fetch_traces(
    filter_string: str | None = None,
    max_results: int = 1000,
    truncate: bool = False,
    exclude: list[str] | None = None,
) -> pd.DataFrame:
    traces = client.search_traces(
        project_name=PROJECT_NAME,
        filter_string=filter_string,
        max_results=max_results,
        truncate=truncate,
        exclude=exclude,
    )
    rows = [as_dict(trace) for trace in traces]
    return pd.json_normalize(rows, sep='.') if rows else pd.DataFrame()

def fetch_threads(
    filter_string: str | None = None,
    max_results: int = 1000,
    truncate: bool = False,
) -> pd.DataFrame:
    threads = client.search_threads(
        project_name=PROJECT_NAME,
        filter_string=filter_string,
        max_results=max_results,
        truncate=truncate,
    )
    rows = [as_dict(thread) for thread in threads]
    return pd.json_normalize(rows, sep='.') if rows else pd.DataFrame()

In [ ]:
trace_df = fetch_traces(max_results=200, truncate=False)
thread_df = fetch_threads(max_results=200, truncate=False)

print('Trace columns:', list(trace_df.columns))
print('Thread columns:', list(thread_df.columns))

if not trace_df.empty:
    print('\nSample trace row (first):')
    print(trace_df.iloc[0].to_dict())

def nested_keys(df: pd.DataFrame, col: str) -> list[str]:
    keys = set()
    if col in df.columns:
        for value in df[col].dropna():
            if isinstance(value, dict):
                keys.update(value.keys())
    return sorted(keys)

print('\nTrace feedback_score keys:', nested_keys(trace_df, 'feedback_scores'))
print('Trace metadata keys:', nested_keys(trace_df, 'metadata'))
print('\nThread feedback_score keys:', nested_keys(thread_df, 'feedback_scores'))


import datetime
from typing import List, Dict, Any

def extract_scores(feedback_scores: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    """
    Extract name, value, and justification (reason) from each feedback score.

    Args:
        feedback_scores: A list of dictionaries, each representing a score.
                         Expected keys: 'name', 'value', 'reason'.

    Returns:
        A list of dictionaries, each containing:
            - 'name': str
            - 'value': float or int
            - 'justification': str (the 'reason' field)
    """
    extracted = []
    for score in feedback_scores:
        extracted.append({
            'name': score.get('name', ''),
            'value': score.get('value', None),
            'justification': score.get('reason', '')  # 'reason' is the justification
        })
    return extracted

In [38]:
import pandas as pd
from typing import Any

TRACE_COLUMNS = [
    'id', 'tags', 'thread_id',
    'metadata.node_id', 'metadata.sender_ids', 'feedback_scores',
    'span_count', 'duration', 'usage.prompt_tokens', 'usage.total_tokens',
]

THREAD_COLUMNS = [
    'id', 'status', 'duration', 'number_of_messages',
    'created_at', 'last_updated_at', 'feedback_scores', 'tags',
]

def select_columns(df: pd.DataFrame, columns: list[str]) -> pd.DataFrame:
    keep = [col for col in columns if col in df.columns]
    return df.loc[:, keep].copy() if keep else df.copy()

def extract_feedback_scores(df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for _, trace in df.iterrows():
        scores = trace.get('feedback_scores')
        if not isinstance(scores, list):
            continue
        for score in scores:
            if not isinstance(score, dict):
                continue
            if score.get('name') not in ['Handoff Quality', 'Responsiveness', 'Role Compliance']:
                continue
            rows.append({
                'trace_id': trace.get('id'),
                'thread_id': trace.get('thread_id'),
                'node_id': trace.get('metadata.node_id'),
                'score_name': score.get('name'),
                'score_value': score.get('value'),
                'score_reason': score.get('reason'),
            })
    scores_df = pd.DataFrame(rows)
    if not scores_df.empty:
        scores_df['score_value'] = pd.to_numeric(scores_df['score_value'], errors='coerce')
    return scores_df

# Fetch
traces_raw = fetch_traces(max_results=200, truncate=False)
threads_raw = fetch_threads(max_results=200, truncate=False)

trace_df = select_columns(traces_raw, TRACE_COLUMNS)
thread_df = select_columns(threads_raw, THREAD_COLUMNS)
feedback_df = extract_feedback_scores(trace_df)

# Mapeamento dos grupos (mantendo os nomes originais com _1, _2, _3)
thread_groups = {
    'client_liaison_run_1': [
        'c629ba35-e4ba-4a1e-b72d-acd350e739d9',
        'ea708e64-6003-4e24-8690-71db753d13f8',
        '2297d6fc-6432-4686-9553-0d19b5ab3d97',
        '9f877593-8e80-4bbf-8179-caa1886cdfdc',
        'd72eda67-643a-4361-aa8e-839a6df93624'
    ],
    'client_liaison_run_2': [
        '738bbc70-f060-4324-ba4b-ef2047771333',
        '8318b44c-288f-4e9d-9599-dc6be3a9c64b',
        '8faf0163-7d38-437d-a84d-9e0974a2d542',
        'd53c91ba-38fb-4704-942e-ac17bf23909c',
        '64f20dae-a7f5-4927-824b-61198103a26e'
    ],
    'client_liaison_run_3': [
        'a404c68f-9395-4412-b4f4-022013abcbe8',
        '95db105e-6e92-4e41-b4fd-b73eba9887e7',
        '4c7a7b65-f852-461e-b2bf-2bdaff50c334',
        'a2a0dfcb-76e1-4de6-8ebc-4077d222bbaf',
        '2c858a77-419e-4931-a6c7-ab2c7fa4cd59'
    ],
    'client_liaison_run_4': [
            'ef06882a-2679-41a8-884b-f8974b0609c9',
            'facd7f24-2191-447a-89d2-545f38cdfe29',
            'ad6878bc-8697-41e6-802e-1d50a15066de',
            '7b7c55f7-7267-4b57-82d8-ee8b595fbd84',
            'ac80fc19-24c9-4942-a486-1fa061f2a176'
    ],

    'client_liaison_run_5': [
        'a3e186b3-2aac-4a26-9391-4e4c926f05ff',
        '050a6bdc-6c14-4b0e-a059-8a4d406558fd',
        '74f97aca-a3f9-4f23-b7b3-a8c1210becb6',
        '3d73e911-705c-481d-90e9-c49a23f6d3b3',
        'fbbc6921-ecaf-48e9-ba98-0c0582af10ae'
    ],
    
    'original_run_1': [
        'd4df6e7c-283e-425a-be5d-c62e6da57870',
        'd77d9d99-e64b-44b6-b949-d107c99f6403',
        '0f0ca6d8-cdf9-4e67-991d-20ed2a621166',
        '52df482c-3ec1-480f-8fd1-6f541c0bccfd',
        '94120fc0-a8e6-4894-a7e1-48c2833afc3e'
    ],
    'original_run_2': [
        '859e0bd9-a02e-4d94-97c7-83738fd26ce6', 
        '77700587-8aa3-43f6-8100-24e834ba7b38', 
        '8f8629df-7631-46d5-a636-173127fc7040', 
        '9ce35d91-0e5b-4b29-ab11-04414d0e3a2c', 
        '37af9b32-161c-4ac3-a452-67e78f53cb12'
    ],
    'original_run_3': [
        'b2b8f898-f3f6-42b3-92be-7e1b6643835e', 
        '092ab63d-3a82-4b5f-9d80-0e8f385a57c5', 
        'c98e0afa-94d3-40ed-8952-d180333f06d4', 
        '9b228c5c-c275-4bc2-adf7-d26bc1441b87', 
        'dbed0c80-02de-4479-91d3-fdb6332e3382'
    ],

    'original_run_6': [
        'fff549e8-1b39-4867-9926-b16a90059bb7',
        'fc94b9a0-9634-446f-9318-95f815363653',
        '951e5e66-3acc-42c7-b9c0-857bc3774789',
        '454e2149-8c00-40f7-ae31-2d0f51f276a4',
        'f1c7f810-4021-4898-a500-213ee03593d0'
    ],

    'original_run_4': [
        '999875b3-1945-441a-bcfa-387467c3fb52', 
        '26a7fb92-1c9f-4c3e-b98f-2d764646b155', 
        'd2ddc9de-5d62-4878-93c1-7a408a3c2431', 
        '1e8fbc46-7e37-4270-8603-2742f9380ff2', 
        '29a3b966-37b5-4536-a53c-df7ed5a8219b'
    ],
    'original_run_5': [
        'ff9e336e-a845-437f-8766-513de78951aa', 
        '8d53a2ce-2e13-44ec-bcf2-3e54315cd645', 
        '8d53a2ce-2e13-44ec-bcf2-3e54315cd645', 
        'b3b5f434-948a-42b0-b1b9-5255a804f6d6', 
        'a338052a-4641-4f4d-a6e1-201a6f3d280e']
}

thread_to_group = {tid: group for group, tids in thread_groups.items() for tid in tids}
selected_threads = list(thread_to_group.keys())

trace_df_filtered = trace_df[trace_df['thread_id'].isin(selected_threads)].copy()
thread_df_filtered = thread_df[thread_df['id'].isin(selected_threads)].copy()

trace_df_filtered['run_group'] = trace_df_filtered['thread_id'].map(thread_to_group)

# Converter tags para string (hashable)
if 'tags' in trace_df_filtered.columns:
    trace_df_filtered['tags'] = trace_df_filtered['tags'].apply(
        lambda x: ', '.join(x) if isinstance(x, list) else str(x)
    )

# Agrupamento SEM trace_count (conforme solicitado: "n deves contabilizar")
grouped_traces = (
    trace_df_filtered
    .groupby(['run_group', 'tags', 'thread_id'], dropna=False)
    .agg(
        trace_ids=('id', list),
        node_ids=('metadata.node_id', list),
        sender_ids=('metadata.sender_ids', list),
        feedback_scores=('feedback_scores', list),
        span_count=('span_count', 'sum'),
        total_duration=('duration', 'sum'),
        avg_duration=('duration', 'mean'),
        prompt_tokens=('usage.prompt_tokens', 'sum'),
        total_tokens=('usage.total_tokens', 'sum'),
    )
    .reset_index()
)

print("Grouped traces (sem contagem de traces):")
print(grouped_traces.head(20))

Grouped traces (sem contagem de traces):
         run_group                       tags                             thread_id  \
0   original_run_3      benchmark_chat_server  b2b8f898-f3f6-42b3-92be-7e1b6643835e   
1   original_run_3     benchmark_csv_pipeline  092ab63d-3a82-4b5f-9d80-0e8f385a57c5   
2   original_run_3  benchmark_expense_tracker  9b228c5c-c275-4bc2-adf7-d26bc1441b87   
3   original_run_3         benchmark_task_api  dbed0c80-02de-4479-91d3-fdb6332e3382   
4   original_run_3    benchmark_url_shortener  c98e0afa-94d3-40ed-8952-d180333f06d4   
5   original_run_4      benchmark_chat_server  d2ddc9de-5d62-4878-93c1-7a408a3c2431   
6   original_run_4     benchmark_csv_pipeline  29a3b966-37b5-4536-a53c-df7ed5a8219b   
7   original_run_4  benchmark_expense_tracker  999875b3-1945-441a-bcfa-387467c3fb52   
8   original_run_4         benchmark_task_api  26a7fb92-1c9f-4c3e-b98f-2d764646b155   
9   original_run_4    benchmark_url_shortener  1e8fbc46-7e37-4270-8603-2742f9380ff2   
10

In [ ]:
import pandas as pd
from typing import Any

# ======================== CORRECTED COLUMNS ========================
# Note: 'feedback_scores' (underscore) not 'feedback scores' (space)
TRACE_COLUMNS = [
    'id',
    'name',
    'tags',
    'thread_id',
    'metadata.node_id',
    'metadata.sender_ids',
    'feedback_scores',          # ✅ fixed column name
    'span_count',
    'duration',
    'usage.prompt_tokens',
    'usage.total_tokens',
]

THREAD_COLUMNS = [
    'id',
    'status',
    'duration',
    'number_of_messages',
    'created_at',
    'last_updated_at',
    'feedback_scores',
    'tags',
]

# ======================== HELPER FUNCTIONS ========================
def select_columns(df: pd.DataFrame, columns: list[str]) -> pd.DataFrame:
    keep = [col for col in columns if col in df.columns]
    return df.loc[:, keep].copy() if keep else df.copy()

def extract_feedback_scores(df: pd.DataFrame) -> pd.DataFrame:
    """
    Extract only name, value, reason from each trace's feedback_scores.
    Also includes trace_id, thread_id, node_id for context.
    """
    rows = []
    for _, trace in df.iterrows():
        scores = trace.get('feedback_scores')
        if not isinstance(scores, list):
            continue
        for score in scores:
            if not isinstance(score, dict):
                continue
            rows.append({
                'trace_id': trace.get('id'),
                'thread_id': trace.get('thread_id'),
                'node_id': trace.get('metadata.node_id'),
                'score_name': score.get('name'),
                'score_value': score.get('value'),
                'score_reason': score.get('reason'),
            })
    scores_df = pd.DataFrame(rows)
    if not scores_df.empty:
        scores_df['score_value'] = pd.to_numeric(scores_df['score_value'], errors='coerce')
    return scores_df

# ======================== FETCH DATA (ONCE) ========================
traces_raw = fetch_traces(max_results=200, truncate=False)
threads_raw = fetch_threads(max_results=200, truncate=False)

trace_df = select_columns(traces_raw, TRACE_COLUMNS)
thread_df = select_columns(threads_raw, THREAD_COLUMNS)

# Extract feedback scores (only name, value, reason)
feedback_df = extract_feedback_scores(trace_df)

print("Feedback scores extracted (first 5 rows):")
print(feedback_df.head())

# ======================== FILTER BY SELECTED THREADS ========================
selected_threads_run_1 = [
    'c629ba35-e4ba-4a1e-b72d-acd350e739d9',
    'ea708e64-6003-4e24-8690-71db753d13f8',
    '2297d6fc-6432-4686-9553-0d19b5ab3d97',
    '9f877593-8e80-4bbf-8179-caa1886cdfdc',
    'd72eda67-643a-4361-aa8e-839a6df93624'
]

selected_threads_run_2 = [
    '738bbc70-f060-4324-ba4b-ef2047771333',
    '8318b44c-288f-4e9d-9599-dc6be3a9c64b',
    '8faf0163-7d38-437d-a84d-9e0974a2d542',
    'd53c91ba-38fb-4704-942e-ac17bf23909c',
    '64f20dae-a7f5-4927-824b-61198103a26e'
]

selected_threads_run_3 = []

selected_threads = selected_threads_run_1 + selected_threads_run_2 + selected_threads_run_3

# Filter traces and threads
trace_df_filtered = trace_df[trace_df['thread_id'].isin(selected_threads)].copy()
thread_df_filtered = thread_df[thread_df['id'].isin(selected_threads)].copy()

print(f"\nFiltered to {len(trace_df_filtered)} traces and {len(thread_df_filtered)} threads.")

# ======================== PREPARE FOR GROUPING ========================
# Convert tags from list to hashable string
if 'tags' in trace_df_filtered.columns:
    trace_df_filtered['tags'] = trace_df_filtered['tags'].apply(
        lambda x: ', '.join(x) if isinstance(x, list) else str(x)
    )

# ======================== GROUP BY TAGS AND THREAD_ID ========================
grouped_traces = (
    trace_df_filtered
    .groupby(['tags', 'thread_id'], dropna=False)
    .agg(
        trace_ids=('id', list),
        trace_names=('name', list),
        node_ids=('metadata.node_id', list),
        sender_ids=('metadata.sender_ids', list),
        trace_count=('id', 'count'),
        span_count=('span_count', 'sum'),
        total_duration=('duration', 'sum'),
        avg_duration=('duration', 'mean'),
        prompt_tokens=('usage.prompt_tokens', 'sum'),
        total_tokens=('usage.total_tokens', 'sum'),
    )
    .reset_index()
)

print("\nGrouped traces (first 20 rows):")
print(grouped_traces.head(20))

# Optional: merge feedback scores per thread/group
feedback_summary = feedback_df.groupby('thread_id').agg({
    'score_name': list,
    'score_value': list,
    'score_reason': list
}).reset_index()

print("\nFeedback scores summary per thread:")
print(feedback_summary.head())

In [25]:
# Inspect and aggregate feedback scores
print('Feedback score columns:', list(feedback_df.columns))
print('Score names:', sorted(feedback_df['score_name'].dropna().unique().tolist()) if not feedback_df.empty else [])

if not feedback_df.empty:
    feedback_summary = feedback_df.groupby('score_name', dropna=False)['score_value'].agg(['count', 'mean', 'min', 'max']).sort_values('mean', ascending=False)
    score_by_role = feedback_df.groupby(['agent_role', 'score_name'], dropna=False)['score_value'].agg(['count', 'mean']).reset_index()
    wide_feedback = feedback_df.pivot_table(index=['trace_id', 'thread_id'], columns='score_name', values='score_value', aggfunc='first').reset_index()
else:
    feedback_summary = pd.DataFrame()
    score_by_role = pd.DataFrame()
    wide_feedback = pd.DataFrame()

feedback_summary, score_by_role.head(20), wide_feedback.head(20)

Feedback score columns: []
Score names: []


(Empty DataFrame
 Columns: []
 Index: [],
 Empty DataFrame
 Columns: []
 Index: [],
 Empty DataFrame
 Columns: []
 Index: [])

## How to select fields

Use `fetch_traces(filter_string=...)` or `fetch_threads(filter_string=...)` with OQL filters.
Examples:
- `thread_id = "thread_123"`
- `metadata.agent_role = "Client Liaison"`
- `feedback_scores.Responsiveness > 0.8`
- `tags contains "production"`

To choose columns, edit `TRACE_COLUMNS` and `THREAD_COLUMNS`, then rerun the selection cells.